In [92]:
import cv2
import mediapipe as mp
import os
import pickle
import asyncio
import numpy as np

In [93]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

In [99]:
video_path = './inputs/'
video_files = []
output_dir = './all_outputs/output_holistic_pickle'

In [102]:
def read_all_files(path):
    # Read all the file names in video_path directory and add them to video_files array
    for filename in os.listdir(path):
        video_files.append(os.path.join(path, filename))

In [96]:
def extract_landmark_data(landmarks):
    """Helper function to convert Mediapipe landmark data into a list of dictionaries."""
    if landmarks:
        return [{'x': lm.x, 'y': lm.y, 'z': lm.z, 'visibility': lm.visibility} for lm in landmarks]
    else:
        return None

In [105]:
def detect(video_path):
    
    video_filename = os.path.basename(video_path)
    output_path = os.path.join(output_dir, f"{os.path.splitext(video_filename)[0]}.pkl")

    # Capture the video
    cap = cv2.VideoCapture(video_path)

    # Structure to store landmarks for each frame
    video_landmarks = []

    with mp_holistic.Holistic(
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as holistic:

        stop = False
        
        while cap.isOpened():
            if stop:
                break
            success, image = cap.read()
            if not success:
                print(f"End of video or empty frame: {video_path}")
                break

            # Convert the image color space from BGR to RGB for processing
            image.flags.writeable = False
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Process the image to get landmarks
            results = holistic.process(image_rgb)

            if results.pose_landmarks:
                
                image_height, image_width, _ = image_rgb.shape
                image_center_x = image_width // 2
                desired_head_y = image_height // 3  # Target head position at 1/3 from the top

                # Calculate the bounding box center of the pose landmarks horizontally
                xs = [lm.x for lm in results.pose_landmarks.landmark]
                ys = [lm.y for lm in results.pose_landmarks.landmark]
                landmark_center_x = int(sum(xs) / len(xs) * image_width)

                # Find the vertical position of the head landmark
                head_landmark = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.NOSE]
                head_y = int(head_landmark.y * image_height)

                # Calculate offsets
                offset_x = image_center_x - landmark_center_x
                offset_y = desired_head_y - head_y

                # Prepare a blank black background for drawing
                blank_image = np.zeros((image_height, image_width, 3), dtype=np.uint8)

                # Adjust pose landmarks by the computed offsets
                for i, landmark in enumerate(results.pose_landmarks.landmark):
                    translated_x = int(landmark.x * image_width + offset_x)
                    translated_y = int(landmark.y * image_height + offset_y)
                    results.pose_landmarks.landmark[i].x = translated_x / image_width
                    results.pose_landmarks.landmark[i].y = translated_y / image_height
                
                # Draw the pose landmarks
                mp_drawing.draw_landmarks(blank_image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

                # Draw left hand landmarks if present
                if results.left_hand_landmarks:
                    for i, landmark in enumerate(results.left_hand_landmarks.landmark):
                        translated_x = int(landmark.x * image_width + offset_x)
                        translated_y = int(landmark.y * image_height + offset_y)
                        results.left_hand_landmarks.landmark[i].x = translated_x / image_width
                        results.left_hand_landmarks.landmark[i].y = translated_y / image_height

                # Draw right hand landmarks if present
                if results.right_hand_landmarks:
                    for i, landmark in enumerate(results.right_hand_landmarks.landmark):
                        translated_x = int(landmark.x * image_width + offset_x)
                        translated_y = int(landmark.y * image_height + offset_y)
                        results.right_hand_landmarks.landmark[i].x = translated_x / image_width
                        results.right_hand_landmarks.landmark[i].y = translated_y / image_height

            # Collect all the landmarks (pose, face, hands) for the image
            frame_landmarks = {
                'pose_landmarks': extract_landmark_data(results.pose_landmarks.landmark) if results.pose_landmarks else None,
                'face_landmarks': extract_landmark_data(results.face_landmarks.landmark) if results.face_landmarks else None,
                'left_hand_landmarks': extract_landmark_data(results.left_hand_landmarks.landmark) if results.left_hand_landmarks else None,
                'right_hand_landmarks': extract_landmark_data(results.right_hand_landmarks.landmark) if results.right_hand_landmarks else None,
            }

            video_landmarks.append(frame_landmarks)

    cap.release()


    # Save landmarks to a pickle file
    with open(output_path, 'wb') as f:
        pickle.dump(video_landmarks, f)

    print(f"Landmarks for video saved as: {output_path}")

In [106]:
read_all_files(video_path)

print(f"Found {len(video_files)} videos")

    # Create the outputs directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
        
    # Process each video one by one
for video_file in video_files:
    print(f"Processing video: {video_file}")
    detect(video_file)
        
print(f"Completed processing {len(video_files)} videos")

Found 12 videos
Processing video: ./inputs/00376.mp4


libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: 

End of video or empty frame: ./inputs/00376.mp4
Landmarks for video saved as: ./all_outputs/output_holistic_pickle/00376.pkl
Processing video: ./inputs/00899.mp4


libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: 

End of video or empty frame: ./inputs/00899.mp4
Landmarks for video saved as: ./all_outputs/output_holistic_pickle/00899.pkl
Processing video: ./inputs/00946.mp4


...

End of video or empty frame: ./inputs/00899.mp4
Landmarks for video saved as: ./all_outputs/output_holistic_pickle/00899.pkl
Processing video: ./inputs/00946.mp4


libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: 

End of video or empty frame: ./inputs/00946.mp4
Landmarks for video saved as: ./all_outputs/output_holistic_pickle/00946.pkl
Processing video: ./inputs/w012.mp4


libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open iris: /usr/lib/dri/iris_dri.so: cannot open shared object file: 

End of video or empty frame: ./inputs/w012.mp4
Landmarks for video saved as: ./all_outputs/output_holistic_pickle/w012.pkl
Completed processing 12 videos
